<a href="https://colab.research.google.com/github/tillyrcohen-code/python-fundamentals/blob/main/TfL_project_flask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install flask requests

In [2]:
import os
os.makedirs("templates", exist_ok=True)

In [3]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en" class="h-full">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>TfL Live Departure Board</title>
    <!-- Tailwind CSS -->
    <script src="https://cdn.tailwindcss.com"></script>
    <!-- FontAwesome Icons -->
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <!-- Google Fonts -->
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap" rel="stylesheet">

    <style>
        body {
            font-family: 'Inter', sans-serif;
            transition: background 0.8s ease, color 0.5s ease;
        }

        /* Weather Themes */
        .theme-default {
            background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
            color: #f8fafc;
        }

        .theme-sunny {
            background: linear-gradient(135deg, #1e1b4b 0%, #311b92 30%, #7c2d12 100%);
            color: #fff7ed;
        }

        .theme-rainy {
            background: linear-gradient(135deg, #091e3a 0%, #11284b 50%, #1e3a8a 100%);
            color: #f0f9ff;
        }

        .theme-cloudy {
            background: #C5C6D0;
            color: #f4f4f5;
        }

        .theme-stormy {
            background: linear-gradient(135deg, #11001c 0%, #220038 50%, #3a005c 100%);
            color: #fae8ff;
        }

        .theme-snowy {
            background: linear-gradient(135deg, #0f172a 0%, #1e293b 50%, #334155 100%);
            color: #f1f5f9;
        }

        /* Glassmorphism Cards */
        .glass-card {
            background: rgba(30, 41, 59, 0.65);
            backdrop-filter: blur(16px);
            -webkit-backdrop-filter: blur(16px);
            border: 1px solid rgba(255, 255, 255, 0.1);
        }

        .glass-input {
            background: rgba(15, 23, 42, 0.75);
            border: 1px solid rgba(255, 255, 255, 0.15);
            color: #f8fafc;
        }

        .glass-input:focus {
            border-color: #38bdf8;
            outline: none;
            box-shadow: 0 0 0 3px rgba(56, 189, 248, 0.2);
        }

        /* Scrollbar */
        ::-webkit-scrollbar {
            width: 6px;
            height: 6px;
        }
        ::-webkit-scrollbar-track {
            background: rgba(0, 0, 0, 0.2);
        }
        ::-webkit-scrollbar-thumb {
            background: rgba(255, 255, 255, 0.2);
            border-radius: 4px;
        }
        ::-webkit-scrollbar-thumb:hover {
            background: rgba(255, 255, 255, 0.4);
        }

        /* Pulse live indicator */
        .pulse-dot {
            box-shadow: 0 0 0 0 rgba(34, 197, 94, 0.7);
            animation: pulse 1.8s infinite;
        }

        @keyframes pulse {
            0% { transform: scale(0.95); box-shadow: 0 0 0 0 rgba(34, 197, 94, 0.7); }
            70% { transform: scale(1); box-shadow: 0 0 0 8px rgba(34, 197, 94, 0); }
            100% { transform: scale(0.95); box-shadow: 0 0 0 0 rgba(34, 197, 94, 0); }
        }

        .collapsible-section {
            max-height: 0;
            overflow: hidden;
            transition: max-height 0.3s ease-out;
        }
        .collapsible-section.expanded {
            max-height: 1000px; /* Arbitrary large value */
            transition: max-height 0.5s ease-in;
        }
    </style>
</head>

<body class="theme-default min-h-full flex flex-col antialiased">
    <div id="app" class="flex-1 max-w-6xl w-full mx-auto p-4 sm:p-6 lg:p-8 flex flex-col gap-6">

        <header class="flex flex-col sm:flex-row justify-between items-start sm:items-center gap-4 border-b border-white/10 pb-5">
            <div class="flex items-center gap-3">
                <div class="w-12 h-12 rounded-xl bg-cyan-500/20 border border-cyan-400/30 flex items-center justify-center text-cyan-400 text-2xl shadow-lg shadow-cyan-500/10">
                    <i class="fa-solid fa-train-subway"></i>
                </div>
                <div>
                    <h1 class="text-2xl sm:text-3xl font-bold tracking-tight text-white flex items-center gap-2">
                        London Transit Board
                    </h1>
                    <p class="text-xs sm:text-sm text-slate-400">Live TfL Departures & Real-time Station Weather</p>
                </div>
            </div>

            <!-- Time & Status Badge -->
            <div class="flex items-center gap-3 self-end sm:self-center">
                <div class="glass-card px-3 py-1.5 rounded-lg flex items-center gap-2 text-xs font-mono text-slate-300">
                    <span class="w-2 h-2 rounded-full bg-emerald-500 pulse-dot"></span>
                    <span id="currentTime">00:00:00</span>
                </div>
            </div>
        </header>

        <section class="glass-card rounded-2xl p-5 sm:p-6 shadow-2xl flex flex-col gap-5">
            <div>
                <label class="block text-xs font-semibold text-slate-400 uppercase tracking-wider mb-2">
                    Quick Line Selection
                </label>
                <div class="flex flex-wrap gap-2" id="linePills">
                    <!-- Dynamic Pill Badges initialised via JavaScript -->
                </div>
            </div>

            <!-- Input Row: Line input & Station dropdown -->
            <div class="grid grid-cols-1 md:grid-cols-12 gap-3 pt-2">

                <!-- Line Input -->
                <div class="md:col-span-5 flex gap-2">
                    <div class="relative flex-1">
                        <i class="fa-solid fa-route absolute left-3.5 top-1/2 -translate-y-1/2 text-slate-400 text-sm"></i>
                        <input type="text" id="lineInput"
                            placeholder="Enter Line (e.g. district, victoria)"
                            class="glass-input w-full pl-10 pr-3 py-2.5 rounded-xl text-sm font-medium transition placeholder:text-slate-500">
                    </div>
                    <button onclick="handleFetchStations()" id="btnFindStations"
                        class="bg-cyan-500 hover:bg-cyan-400 text-slate-950 font-semibold px-4 py-2.5 rounded-xl text-sm transition-all flex items-center gap-2 whitespace-nowrap active:scale-95 shadow-lg shadow-cyan-500/20">
                        <i class="fa-solid fa-magnifying-glass"></i>
                        <span>Find Stations</span>
                    </button>
                </div>

                <!-- Station Dropdown -->
                <div class="md:col-span-5 relative">
                    <i class="fa-solid fa-location-dot absolute left-3.5 top-1/2 -translate-y-1/2 text-slate-400 text-sm z-10"></i>
                    <select id="stationSelect" disabled
                        class="glass-input w-full pl-10 pr-8 py-2.5 rounded-xl text-sm font-medium transition appearance-none cursor-pointer disabled:opacity-50 disabled:cursor-not-allowed">
                        <option value="">-- Enter a line first --</option>
                    </select>
                    <i class="fa-solid fa-chevron-down absolute right-3.5 top-1/2 -translate-y-1/2 text-slate-400 text-xs pointer-events-none"></i>
                </div>

                <!-- Search Departures Button -->
                <div class="md:col-span-2">
                    <button onclick="handleFetchDepartures()" id="btnSearch" disabled
                        class="w-full bg-emerald-500 hover:bg-emerald-400 disabled:bg-slate-700 disabled:text-slate-500 text-slate-950 font-semibold px-4 py-2.5 rounded-xl text-sm transition-all flex items-center justify-center gap-2 active:scale-95 disabled:pointer-events-none shadow-lg shadow-emerald-500/10">
                        <i class="fa-solid fa-clock"></i>
                        <span>Departures</span>
                    </button>
                </div>
            </div>

            <!-- Status Message Bar -->
            <div id="statusNotification" class="hidden text-xs px-3 py-2 rounded-lg flex items-center gap-2">
                <i id="statusIcon" class="fa-solid fa-info-circle"></i>
                <span id="statusText">Ready</span>
            </div>
        </section>

        <section id="stationWeatherCard" class="hidden glass-card rounded-2xl p-5 sm:p-6 shadow-xl flex flex-col md:flex-row justify-between items-start md:items-center gap-4 border-l-4 border-l-cyan-400">
            <div class="flex items-center gap-4">
                <div id="weatherIconContainer" class="w-14 h-14 rounded-2xl bg-white/10 flex items-center justify-center text-3xl shadow-inner">
                    🌡️
                </div>
                <div>
                    <div class="flex items-center gap-2">
                        <span id="activeLineBadge" class="text-xs px-2.5 py-0.5 rounded-full font-bold uppercase bg-cyan-500/20 text-cyan-300 border border-cyan-500/30">
                            LINE
                        </span>
                        <h2 id="activeStationTitle" class="text-xl sm:text-2xl font-bold text-white">Station Name</h2>
                    </div>
                    <p id="weatherDescriptionText" class="text-xs sm:text-sm text-slate-300 mt-1 flex items-center gap-2">
                        <span>Weather data loading...</span>
                    </p>
                </div>
            </div>

            <!-- Auto Refresh Control -->
            <div class="flex items-center gap-3 self-end md:self-center bg-black/20 px-3.5 py-2 rounded-xl border border-white/5">
                <div class="text-right">
                    <p class="text-[10px] text-slate-400 uppercase tracking-wider font-semibold">Auto Refresh</p>
                    <p id="countdownTimer" class="text-xs font-mono font-bold text-cyan-400">30s</p>
                </div>
                <button onclick="toggleAutoRefresh()" id="btnAutoRefresh" class="w-8 h-8 rounded-lg bg-white/10 hover:bg-white/20 text-slate-200 flex items-center justify-center transition">
                    <i class="fa-solid fa-pause text-xs"></i>
                </button>
            </div>
        </section>

        <main class="glass-card rounded-2xl p-5 sm:p-6 shadow-2xl flex-1 flex flex-col">
            <div class="flex justify-between items-center mb-4">
                <h3 class="text-lg font-bold text-white flex items-center gap-2">
                    <i class="fa-solid fa-list-check text-cyan-400"></i>
                    <span>Live Departures</span>
                </h3>
                <span id="departureCount" class="text-xs font-medium text-slate-400 bg-white/5 px-2.5 py-1 rounded-full">
                    0 trains predicted
                </span>
            </div>

            <!-- Departures Table Wrapper -->
            <div class="overflow-x-auto flex-1">
                <table class="w-full text-left border-collapse">
                    <thead>
                        <tr class="border-b border-white/10 text-xs font-semibold text-slate-400 uppercase tracking-wider">
                            <th class="py-3 px-4">Line / Operator</th>
                            <th class="py-3 px-4">Destination</th>
                            <th class="py-3 px-4">Platform</th>
                            <th class="py-3 px-4 text-right">Expected Arrival</th>
                        </tr>
                    </thead>
                    <tbody id="departuresTableBody" class="divide-y divide-white/5 text-sm">
                        <tr>
                            <td colspan="4" class="py-12 text-center text-slate-400">
                                <div class="flex flex-col items-center gap-3">
                                    <i class="fa-solid fa-train text-3xl text-slate-600"></i>
                                    <p class="font-medium text-slate-300">No station selected yet.</p>
                                    <p class="text-xs text-slate-500">Enter a line above, choose a station from the list, and hit Departures!</p>
                                </div>
                            </td>
                        </tr>
                    </tbody>
                </table>
            </div>

            <!-- Button to show other lines -->
            <div id="otherLinesToggleContainer" class="mt-4 text-center hidden">
                <button onclick="toggleOtherLines()" id="btnToggleOtherLines"
                    class="bg-blue-500 hover:bg-blue-400 text-white font-semibold px-4 py-2 rounded-xl text-sm transition-all flex items-center justify-center gap-2 mx-auto active:scale-95 shadow-lg shadow-blue-500/20">
                    <i class="fa-solid fa-eye"></i>
                    <span>Show Other Lines At This Station</span>
                </button>
            </div>

            <!-- Other Lines Departures Section -->
            <div id="otherLinesDeparturesSection" class="collapsible-section mt-4">
                <h4 class="text-md font-bold text-white mb-3">Departures on Other Lines</h4>
                <table class="w-full text-left border-collapse">
                    <thead>
                        <tr class="border-b border-white/10 text-xs font-semibold text-slate-400 uppercase tracking-wider">
                            <th class="py-3 px-4">Line / Operator</th>
                            <th class="py-3 px-4">Destination</th>
                            <th class="py-3 px-4">Platform</th>
                            <th class="py-3 px-4 text-right">Expected Arrival</th>
                        </tr>
                    </thead>
                    <tbody id="otherDeparturesTableBody" class="divide-y divide-white/5 text-sm">
                        <!-- Other lines will be loaded here -->
                    </tbody>
                </table>
            </div>

        </main>

        <footer class="text-center text-xs text-slate-500 py-2 flex flex-col sm:flex-row justify-between items-center gap-2 border-t border-white/5">
            <p>Data powered by <span class="text-slate-400 font-semibold">TfL Unified API</span> & <span class="text-slate-400 font-semibold">Open-Meteo</span></p>
            <p>London Transit Board Dashboard &copy; Live Transit Sync</p>
        </footer>
    </div>

    <script>
        // Pre-defined popular lines for quick selection
        const POPULAR_LINES = [
            { id: 'district', name: 'District', color: 'border-emerald-500/40 text-emerald-300 bg-emerald-500/10' },
            { id: 'victoria', name: 'Victoria', color: 'border-sky-500/40 text-sky-300 bg-sky-500/10' },
            { id: 'northern', name: 'Northern', color: 'border-slate-400/40 text-slate-200 bg-slate-400/10' },
            { id: 'central', name: 'Central', color: 'border-red-500/40 text-red-300 bg-red-500/10' },
            { id: 'jubilee', name: 'Jubilee', color: 'border-slate-400/40 text-slate-300 bg-slate-500/10' },
            { id: 'piccadilly', name: 'Piccadilly', color: 'border-blue-600/40 text-blue-300 bg-blue-600/10' },
            { id: 'elizabeth', name: 'Elizabeth', color: 'border-fuchsia-500/40 text-fuchsia-300 bg-fuchsia-500/10' }
        ];

        // Global State
        let currentStationsList = [];
        let selectedStationObj = null;
        let activeLineId = '';
        let autoRefreshInterval = null;
        let countdownValue = 30;
        let isPaused = false;

        document.addEventListener('DOMContentLoaded', () => {
            initClock();
            renderLinePills();

            // Handle enter key on input
            document.getElementById('lineInput').addEventListener('keypress', (e) => {
                if (e.key === 'Enter') {
                    handleFetchStations();
                }
            });

            // Enable search button when station is selected
            document.getElementById('stationSelect').addEventListener('change', (e) => {
                const stationId = e.target.value;
                const btnSearch = document.getElementById('searchBtn') || document.getElementById('btnSearch');

                if (stationId) {
                    selectedStationObj = currentStationsList.find(s => s.id === stationId);
                    btnSearch.disabled = false;
                } else {
                    selectedStationObj = null;
                    btnSearch.disabled = true;
                }
            });
        });

        // Real-time Clock
        function initClock() {
            const clockEl = document.getElementById('currentTime');
            const update = () => {
                const now = new Date();
                clockEl.innerText = now.toLocaleTimeString('en-GB');
            };
            update();
            setInterval(update, 1000);
        }

        // Render Quick Select
        function renderLinePills() {
            const container = document.getElementById('linePills');
            container.innerHTML = POPULAR_LINES.map(line => `
                <button onclick="quickSelectLine('${line.id}')"
                    class="text-xs px-3 py-1.5 rounded-lg border font-semibold transition-all active:scale-95 hover:brightness-125 ${line.color}">
                    ${line.name}
                </button>
            `).join('');
        }

        function quickSelectLine(lineId) {
            document.getElementById('lineInput').value = lineId;
            handleFetchStations();
        }

        function showStatus(message, type = 'info') {
            const box = document.getElementById('statusNotification');
            const text = document.getElementById('statusText');
            const icon = document.getElementById('statusIcon');

            box.classList.remove('hidden', 'bg-red-500/20', 'text-red-300', 'border-red-500/30',
                                'bg-emerald-500/20', 'text-emerald-300', 'border-emerald-500/30',
                                'bg-cyan-500/20', 'text-cyan-300', 'border-cyan-500/30');

            if (type === 'error') {
                box.classList.add('bg-red-500/20', 'text-red-300', 'border', 'border-red-500/30');
                icon.className = 'fa-solid fa-triangle-exclamation';
            } else if (type === 'success') {
                box.classList.add('bg-emerald-500/20', 'text-emerald-300', 'border', 'border-emerald-500/30');
                icon.className = 'fa-solid fa-check-circle';
            } else {
                box.classList.add('bg-cyan-500/20', 'text-cyan-300', 'border', 'border-cyan-500/30');
                icon.className = 'fa-solid fa-circle-notch fa-spin';
            }

            text.innerHTML = message;
        }

        function hideStatus() {
            document.getElementById('statusNotification').classList.add('hidden');
        }

        async function handleFetchStations() {
            const lineInput = document.getElementById('lineInput').value.trim().toLowerCase();
            const stationSelect = document.getElementById('stationSelect');
            const btnFind = document.getElementById('btnFindStations');
            const btnSearch = document.getElementById('btnSearch');

            if (!lineInput) {
                showStatus('Please enter a transit line name (e.g. district, victoria)', 'error');
                return;
            }

            activeLineId = lineInput;
            btnFind.disabled = true;
            btnFind.innerHTML = `<i class="fa-solid fa-spinner fa-spin"></i> Loading...`;
            stationSelect.disabled = true;
            stationSelect.innerHTML = `<option value="">Loading stations for '${lineInput}'...</option>`;
            btnSearch.disabled = true;
            showStatus(`Querying TfL API for line: <strong>${lineInput}</strong>...`);

            try {
                const response = await fetch(`https://api.tfl.gov.uk/Line/${encodeURIComponent(lineInput)}/StopPoints`);

                if (!response.ok) {
                    throw new Error(`Line '${lineInput}' not found on TfL network.`);
                }

                const data = await response.json();

                if (!data || data.length === 0) {
                    throw new Error(`No stations found for line '${lineInput}'.`);
                }

                // Deduplicate stations by name and extract needed data
                const uniqueStationsMap = new Map();
                data.forEach(stop => {
                    const cleanName = stop.commonName.replace(/ Underground Station| Rail Station| DLR Station/gi, '').trim();
                    if (!uniqueStationsMap.has(cleanName)) {
                        uniqueStationsMap.set(cleanName, {
                            id: stop.id,
                            name: cleanName,
                            rawName: stop.commonName,
                            lat: stop.lat,
                            lon: stop.lon
                        });
                    }
                });

                currentStationsList = Array.from(uniqueStationsMap.values()).sort((a, b) => a.name.localeCompare(b.name));

                // Populate Select Dropdown
                stationSelect.innerHTML = `<option value="">-- Choose a station (${currentStationsList.length} available) --</option>` +
                    currentStationsList.map(s => `<option value="${s.id}">${s.name}</option>`).join('');

                stationSelect.disabled = false;
                showStatus(`Successfully loaded <strong>${currentStationsList.length}</strong> stations on ${lineInput.toUpperCase()} line. Select a station to view live board.`, 'success');

            } catch (err) {
                showStatus(err.message || 'Failed to fetch stations. Please check the line name.', 'error');
                stationSelect.innerHTML = `<option value="">-- Line not found --</option>`;
            } finally {
                btnFind.disabled = false;
                btnFind.innerHTML = `<i class="fa-solid fa-magnifying-glass"></i><span>Find Stations</span>`;
            }
        }

        async function handleFetchDepartures() {
            if (!selectedStationObj) return;

            const btnSearch = document.getElementById('btnSearch');
            btnSearch.disabled = true;
            showStatus(`Fetching live train arrivals & weather for <strong>${selectedStationObj.name}</strong>...`);

            // Start or restart countdown
            resetCountdown();

            try {
                // Fetch Arrivals and Weather concurrently
                const [arrivalsRes, weatherRes] = await Promise.allSettled([
                    fetch(`/api/departures?stop_point_id=${selectedStationObj.id}`), // Fetch all arrivals
                    fetchWeather(selectedStationObj.lat, selectedStationObj.lon)
                ]);

                let departuresPayload = {};
                if (arrivalsRes.status === 'fulfilled' && arrivalsRes.value.ok) {
                    departuresPayload = await arrivalsRes.value.json();
                }

                let weatherData = null;
                if (weatherRes.status === 'fulfilled') {
                    weatherData = weatherRes.value;
                }

                renderDeparturesTable(departuresPayload.departures || []);
                renderWeatherWidget(weatherData);

                showStatus(`Live board updated for <strong>${selectedStationObj.name}</strong>`, 'success');

            } catch (err) {
                showStatus('Error updating departure board.', 'error');
            } finally {
                btnSearch.disabled = false;
            }
        }

        async function fetchWeather(lat, lon) {
            if (!lat || !lon) return null;

            try {
                const res = await fetch(`https://api.open-meteo.com/v1/forecast?latitude=${lat}&longitude=${lon}&current_weather=true`);
                if (!res.ok) return null;
                const data = await res.json();

                const current = data.current_weather;
                const temp = Math.round(current.temperature);
                const code = current.weathercode;

                // Weather code mapping
                const weatherMap = {
                    0: { emoji: '☀️', text: 'Clear Sky', theme: 'theme-sunny' },
                    1: { emoji: '🌤️', text: 'Mainly Clear', theme: 'theme-sunny' },
                    2: { emoji: '⛅', text: 'Partly Cloudy', theme: 'theme-cloudy' },
                    3: { emoji: '☁️', text: 'Overcast', theme: 'theme-cloudy' },
                    45: { emoji: '🌫️', text: 'Foggy', theme: 'theme-cloudy' },
                    48: { emoji: '🌫️', text: 'Freezing Fog', theme: 'theme-cloudy' },
                    51: { emoji: '🌧️', text: 'Light Drizzle', theme: 'theme-rainy' },
                    61: { emoji: '🌧️', text: 'Rainy', theme: 'theme-rainy' },
                    63: { emoji: '🌧️', text: 'Moderate Rain', theme: 'theme-rainy' },
                    71: { emoji: '❄️', text: 'Snowy', theme: 'theme-snowy' },
                    95: { emoji: '⛈️', text: 'Thunderstorm', theme: 'theme-stormy' }
                };

                const info = weatherMap[code] || { emoji: '🌡️', text: 'Weather Active', theme: 'theme-default' };
                return {
                    temp: temp,
                    wind: current.windspeed,
                    ...info
                };
            } catch (e) {
                return null;
            }
        }

        function renderWeatherWidget(w) {
            const card = document.getElementById('stationWeatherCard');
            card.classList.remove('hidden');

            document.getElementById('activeStationTitle').innerText = selectedStationObj.name;
            document.getElementById('activeLineBadge').innerText = activeLineId.toUpperCase();

            const weatherIcon = document.getElementById('weatherIconContainer');
            const weatherText = document.getElementById('weatherDescriptionText');

            if (w) {
                weatherIcon.innerText = w.emoji;
                weatherText.innerHTML = `<span class="font-bold text-white">${w.temp}°C</span> • ${w.text} • Wind: ${w.wind} km/h`;
                applyBodyTheme(w.theme);
            } else {
                weatherIcon.innerText = '🌡️';
                weatherText.innerText = 'Weather data unavailable';
                applyBodyTheme('theme-default');
            }
        }

        function applyBodyTheme(themeClass) {
            document.body.className = `${themeClass} min-h-full flex flex-col antialiased`;
        }

        function formatDeparture(dep) {
            const mins = Math.floor(dep.timeToStation / 60);
            const isDue = mins <= 0;
            const timeBadge = isDue
                ? `<span class="px-2.5 py-1 rounded-lg text-xs font-bold bg-emerald-500/20 text-emerald-300 border border-emerald-500/30 pulse-dot">Due</span>`
                : `<span class="text-base font-bold text-cyan-300">${mins} <span class="text-xs font-normal text-slate-400">min</span></span>`;

            return `
                <tr class="hover:bg-white/5 transition-colors">
                    <td class="py-3.5 px-4 font-semibold text-white flex items-center gap-2">
                        <span class="w-2 h-2 rounded-full bg-cyan-400"></span>
                        <span>${dep.lineName || 'Unknown Line'}</span>
                    </td>
                    <td class="py-3.5 px-4 font-medium text-slate-200">
                        ${dep.destinationName || 'Destination Unknown'}
                        ${dep.towards ? `<span class="block text-xs text-slate-500 font-normal">Towards ${dep.towards}</span>` : ''}
                    </td>
                    <td class="py-3.5 px-4 text-slate-300 text-xs">
                        ${dep.platformName || 'Plat. N/A'}
                    </td>
                    <td class="py-3.5 px-4 text-right">
                        ${timeBadge}
                    </td>
                </tr>
            `;
        }

        function renderDeparturesTable(allDepartures) {
            const tbody = document.getElementById('departuresTableBody');
            const otherTbody = document.getElementById('otherDeparturesTableBody');
            const countBadge = document.getElementById('departureCount');
            const otherLinesToggleContainer = document.getElementById('otherLinesToggleContainer');
            const otherLinesDeparturesSection = document.getElementById('otherLinesDeparturesSection');

            // Reset visibility
            otherLinesToggleContainer.classList.add('hidden');
            otherLinesDeparturesSection.classList.remove('expanded');
            otherTbody.innerHTML = ''; // Clear previous other lines

            const departuresArray = Array.isArray(allDepartures) ? allDepartures : [];

            if (departuresArray.length === 0) {
                countBadge.innerText = '0 trains predicted';
                tbody.innerHTML = `
                    <tr>
                        <td colspan="4" class="py-12 text-center text-slate-400">
                            <div class="flex flex-col items-center gap-2">
                                <i class="fa-solid fa-circle-exclamation text-amber-400 text-2xl"></i>
                                <p class="font-medium text-slate-200">No live departures found right now.</p>
                                <p class="text-xs text-slate-500">Service might be paused, or this may be a commuter station with no TfL lines.</p>
                            </div>
                        </td>
                    </tr>`;
                return;
            }

            // Separate departures for the active line and other lines
            const selectedLineDepartures = departuresArray.filter(
                dep => dep.lineId && dep.lineId.toLowerCase() === activeLineId.toLowerCase()
            ).sort((a, b) => a.timeToStation - b.timeToStation);

            const otherLineDepartures = departuresArray.filter(
                dep => dep.lineId && dep.lineId.toLowerCase() !== activeLineId.toLowerCase()
            );

            // Render selected line departures (top 6)
            const topSelectedLineDepartures = selectedLineDepartures.slice(0, 6);
            countBadge.innerText = `${topSelectedLineDepartures.length} trains predicted`;
            tbody.innerHTML = topSelectedLineDepartures.map(formatDeparture).join('');

            // Process other lines
            if (otherLineDepartures.length > 0) {
                const otherLinesGrouped = otherLineDepartures.reduce((acc, dep) => {
                    if (!acc[dep.lineId]) {
                        acc[dep.lineId] = [];
                    }
                    acc[dep.lineId].push(dep);
                    return acc;
                }, {});

                let otherLinesHtml = '';
                for (const lineId in otherLinesGrouped) {
                    const lineSpecificDepartures = otherLinesGrouped[lineId]
                        .sort((a, b) => a.timeToStation - b.timeToStation)
                        .slice(0, 2); // Top 2 for each other line

                    if (lineSpecificDepartures.length > 0) {
                        otherLinesHtml += `
                            <tr><td colspan="4" class="py-2 px-4 text-xs font-bold uppercase tracking-wider text-slate-400 bg-white/5">${lineSpecificDepartures[0].lineName || lineId.toUpperCase()}</td></tr>
                            ` + lineSpecificDepartures.map(formatDeparture).join('');
                    }
                }

                if (otherLinesHtml) {
                    otherLinesToggleContainer.classList.remove('hidden');
                    otherTbody.innerHTML = otherLinesHtml;
                }
            }

            if (topSelectedLineDepartures.length === 0 && otherLineDepartures.length === 0) {
                 tbody.innerHTML = `
                    <tr>
                        <td colspan="4" class="py-12 text-center text-slate-400">
                            <div class="flex flex-col items-center gap-2">
                                <i class="fa-solid fa-circle-exclamation text-amber-400 text-2xl"></i>
                                <p class="font-medium text-slate-200">No live departures found for any line at this station right now.</p>
                                <p class="text-xs text-slate-500">Service might be paused, or this may be a commuter station with no TfL lines.</p>
                            </div>
                        </td>
                    </tr>`;
            }
        }

        function toggleOtherLines() {
            const otherLinesDeparturesSection = document.getElementById('otherLinesDeparturesSection');
            const btnToggleOtherLines = document.getElementById('btnToggleOtherLines');

            if (otherLinesDeparturesSection.classList.contains('expanded')) {
                otherLinesDeparturesSection.classList.remove('expanded');
                btnToggleOtherLines.innerHTML = `<i class="fa-solid fa-eye"></i><span>Show Other Lines At This Station</span>`;
            } else {
                otherLinesDeparturesSection.classList.add('expanded');
                btnToggleOtherLines.innerHTML = `<i class="fa-solid fa-eye-slash"></i><span>Hide Other Lines</span>`;
            }
        }

        function resetCountdown() {
            clearInterval(autoRefreshInterval);
            countdownValue = 30;
            updateCountdownDisplay();

            if (!isPaused) {
                autoRefreshInterval = setInterval(() => {
                    countdownValue--;
                    updateCountdownDisplay();

                    if (countdownValue <= 0) {
                        countdownValue = 30;
                        handleFetchDepartures();
                    }
                }, 1000);
            }
        }

        function updateCountdownDisplay() {
            document.getElementById('countdownTimer').innerText = isPaused ? 'Paused' : `${countdownValue}s`;
        }

        function toggleAutoRefresh() {
            isPaused = !isPaused;
            const btn = document.getElementById('btnAutoRefresh');
            btn.innerHTML = isPaused ? `<i class="fa-solid fa-play text-xs"></i>` : `<i class="fa-solid fa-pause text-xs"></i>`;
            resetCountdown();
        }
    </script>
</body>
</html>

Writing templates/index.html


In [4]:
%%writefile app.py
from flask import Flask, render_template, request, jsonify
import requests
import re

app = Flask(__name__)

def get_station_weather(location_name):
    try:
        clean_name = re.sub(r'\(.*?\)', '', location_name)
        for keyword in ["Rail Station", "Underground Station", "DLR Station", "Station"]:
            clean_name = clean_name.replace(keyword, "")
        clean_name = clean_name.strip()

        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={clean_name}&count=1&language=en&format=json&country_code=gb"
        geo_res = requests.get(geo_url, timeout=5).json()

        if not geo_res.get("results"):
            return {"temp": "--", "emoji": "🌡️", "description": "Unavailable"}

        station_data = geo_res["results"][0]
        lat = station_data["latitude"]
        lon = station_data["longitude"]

        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        w_res = requests.get(weather_url, timeout=5).json()

        current = w_res.get("current_weather", {})
        temp = round(current.get("temperature", 0))
        code = current.get("weathercode", 0)

        weather_map = {
            0: ("☀️", "Clear Sky"), 1: ("🌤️", "Mainly Clear"), 2: ("⛅", "Partly Cloudy"),
            3: ("☁️", "Overcast"), 45: ("🌫️", "Foggy"), 51: ("🌧️", "Light Drizzle"),
            61: ("🌧️", "Slight Rain"), 63: ("🌧️", "Moderate Rain"), 95: ("⛈️", "Thunderstorm")
        }

        emoji, desc = weather_map.get(code, ("🌡️", "Weather Active"))
        return {"temp": temp, "emoji": emoji, "description": desc}

    except Exception:
        return {"temp": "--", "emoji": "🌡️", "description": "Unavailable"}

@app.route("/")
def index():
    return render_template("index.html")

@app.route("/api/stations")
def get_stations():
    line = request.args.get("line", "").strip().lower()
    if not line:
        return jsonify({"error": "Line parameter is required"}), 400

    try:
        url = f"https://api.tfl.gov.uk/Line/{line}/StopPoints"
        res = requests.get(url, timeout=5)

        if res.status_code != 200:
            return jsonify({"error": f"Line '{line}' not found. Try 'district', or 'victoria'."}), 404

        data = res.json()
        stations = []
        for stop in data:
            name = stop.get("commonName", "")
            if name:
                stations.append({"name": name, "id": stop.get("id")}) # Include ID

        unique_stations = sorted(list({s["name"]: s for s in stations}.values()), key=lambda x: x["name"])
        return jsonify({"line": line, "stations": unique_stations})

    except Exception as e:
        return jsonify({"error": f"Failed to fetch stations: {str(e)}"}), 500

@app.route("/api/departures")
def get_departures():
    stop_point_id = request.args.get("stop_point_id", "").strip()

    if not stop_point_id:
        return jsonify({"error": "Stop Point ID is required"}), 400

    try:
        # Fetch station details to get name, lat, lon for weather
        station_details_url = f"https://api.tfl.gov.uk/StopPoint/{stop_point_id}"
        station_details_res = requests.get(station_details_url, timeout=5)
        station_details = station_details_res.json()

        station_name = station_details.get("commonName", "Unknown Station")
        lat = station_details.get("lat")
        lon = station_details.get("lon")

        weather_data = get_station_weather(station_name) if lat and lon else {"temp": "--", "emoji": "🌡️", "description": "Unavailable"}

        url = f"https://api.tfl.gov.uk/StopPoint/{stop_point_id}/Arrivals"
        res = requests.get(url, timeout=5)

        all_arrivals = []
        if res.status_code == 200:
            arrivals = res.json()
            for arr in arrivals:
                all_arrivals.append({
                    "lineId": arr.get("lineId"),
                    "lineName": arr.get("lineName"),
                    "destinationName": arr.get("destinationName"),
                    "platformName": arr.get("platformName"),
                    "timeToStation": arr.get("timeToStation"),
                    "towards": arr.get("towards")
                })

        # No filtering or sorting here; return all raw arrivals to frontend
        return jsonify({
            "station_name": station_name,
            "source": "TfL Live API",
            "weather": weather_data,
            "departures": all_arrivals
        })

    except Exception as e:
        return jsonify({"error": f"Failed to fetch departures: {str(e)}"}), 500

if __name__ == "__main__":
    app.run(port=5000)

Writing app.py


In [ ]:
from google.colab.output import eval_js

print("Click this link to open your live web dashboard:")
print(eval_js("google.colab.kernel.proxyPort(5000)"))

!python app.py

Click this link to open your live web dashboard:
https://5000-m-s-usw4c0-vq5dvv7an17k-c.us-west4.prod.r.colab.dev
 * Serving Flask app 'app'
 * Debug mode: off
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [24/Nov/2026 13:40:59] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [24/Nov/2026 13:41:00] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [24/Nov/2026 13:41:10] "GET /api/departures?stop_point_id=940GZZLUSVS HTTP/1.1" 200 -
127.0.0.1 - - [24/Nov/2026 13:41:41] "GET /api/departures?stop_point_id=940GZZLUSVS HTTP/1.1" 200 -
127.0.0.1 - - [24/Nov/2026 13:42:21] "GET /api/departures?stop_point_id=940GZZLUSVS HTTP/1.1" 200 -
127.0.0.1 - - [24/Nov/2026 13:52:15] "GET /api/departures?stop_point_id=940GZZLUSVS HTTP/1.1" 200 -
127.0.0.1 - - [24/Nov/2026 13:54:44] "GET /api/departures?stop_point_id=940GZZLUSVS HTTP/1.1" 200 -
127.0.0.1 - - [24/Nov/2026 13:56:30] "GET /api/departures?stop_point_id=940GZZLUSVS HTTP/1.1" 500 -
